In [1]:
import json 
import pandas as pd
import numpy as np
import os

In [2]:
def flatten_episode_complete(episode):
    """Extract and flatten study_data and conversation fields for JSONL format"""
    flattened = {}
    study_data = episode['study_data']
    
    # Copy basic fields (episode_id, basic_info)
    for key, value in episode.items():
        if key in ['episode_id', 'basic_info']:
            flattened[key] = value
    
    # Extract direct fields from study_data
    direct_fields = ['scenario_id', 'agent_id']
    for field in direct_fields:
        if field in study_data:
            flattened[field] = study_data[field]
    
    # Flatten conversation_stats
    flattened['conv_length'] = study_data.get('conversation_stats', {}).get('total_turns')
    
    # Flatten prolific_data
    if 'prolific_data' in study_data:
        for field, value in study_data['prolific_data'].items():
            flattened[field] = value
    
    # Flatten survey_responses
    if 'survey_responses' in study_data:
        for response, value in study_data['survey_responses'].items():
            flattened[f'survey_{response}'] = value
    
    # Flatten personality_assessment
    if 'personality_assessment' in study_data:
        personality = study_data['personality_assessment']
        
        # Scores
        if 'scores' in personality:
            for score_type, score_value in personality['scores'].items():
                flattened[f'personality_score_{score_type}'] = score_value
        
        # Classification
        if 'classification' in personality:
            for class_type, class_value in personality['classification'].items():
                flattened[f'personality_{class_type}'] = class_value
    
    # === FLATTEN CONVERSATION ===
    if 'conversation' in episode:
        conversation = episode['conversation']
        
        # Extract and flatten messages
        if 'messages' in conversation:
            messages = conversation['messages']
            flattened_messages = []
            
            for turn_idx, turn in enumerate(messages):
                for message_idx, message in enumerate(turn):
                    flattened_messages.append({
                        'turn': turn_idx + 1,
                        'message_in_turn': message_idx + 1,
                        'speaker': message[0],
                        'action_type': message[1],
                        'content': message[2]
                    })
            
            flattened['messages'] = flattened_messages
    
    return flattened

In [42]:
with open('user_study_enhanced_20250921_204509.json', 'r') as file:
    data = json.load(file)
print(data.keys())

# filtered_episodes = []
# for episode in data['episodes']:
#     study_data = episode.get('study_data', {})
#     if study_data['prolific_data']['PROLIFIC_PID'] is not None:
#         filtered_episodes.append(episode)
    
# raw_df = pd.DataFrame(filtered_episodes)

dict_keys(['export_info', 'statistics', 'episodes'])


In [49]:
data['episodes'][0]['metadata']

{'rewards': [0.0, 0.0],
 'raw_reasoning': '{\n  "study_type": "user_study_human_ai_conversation",\n  "session_id": "user_study_20250901_171525_79b60c34",\n  "timestamp": "2025-09-01T17:15:25.715906",\n  "interventions": {\n    "transparency": "high",\n    "warmth": "high",\n    "expertise": "high",\n    "adaptability": "high",\n    "theory_of_mind": "high"\n  },\n  "scenario_id": "job_interview_competitive",\n  "agent_id": "AI Agent (Hiring Manager) - Unknown_HighT_HighW",\n  "agent_attributes": {\n    "name": "AI Agent",\n    "occupation": "Hiring Manager",\n    "age": 22,\n    "decision_making_style": "",\n    "big_five": "",\n    "mbti": ""\n  },\n  "conversation_stats": {\n    "total_turns": 2,\n    "human_messages": 1,\n    "ai_messages": 1,\n    "avg_message_length": 335.0\n  },\n  "prolific_data": {\n    "PROLIFIC_PID": null,\n    "STUDY_ID": null,\n    "SESSION_ID": null\n  },\n  "survey_responses": {\n    "transparency": 1,\n    "warmth": 1,\n    "theory_of_mind": 1,\n    "ada

In [51]:
data['episodes'][0]['basic_info']

{'tag': 'user_study_20250901_171525_79b60c34',
 'environment': 'job_interview_competitive',
 'agents': ['AI Agent (Hiring Manager) - Unknown_HighT_HighW',
  'human_participant'],
 'models': ['transparency_high',
  'warmth_high',
  'expertise_high',
  'adaptability_high',
  'theory_of_mind_high',
  'human']}

In [53]:
# Test the complete function
test_episode = data['episodes'][10]
flattened_complete = flatten_episode_complete(test_episode)
flattened_complete

{'episode_id': '01K5CRE3ARWKEB41G8XC4NWBW9',
 'basic_info': {'tag': 'user_study_20250917_171407_97832dbc',
  'environment': 'job_interview_competitive',
  'agents': ['AI Agent (Hiring Manager) - LowT_LowW_HighE_HighA_HighTOM',
   'human_participant'],
  'models': ['transparency_low',
   'warmth_low',
   'expertise_high',
   'adaptability_high',
   'theory_of_mind_high',
   'human']},
 'scenario_id': 'job_interview_competitive',
 'agent_id': 'AI Agent (Hiring Manager) - LowT_LowW_HighE_HighA_HighTOM',
 'conv_length': 13,
 'PROLIFIC_PID': '66d76d7e0523ccc0cd1a8606',
 'STUDY_ID': '68c0736497f9e985c62bac05',
 'SESSION_ID': '68cb21fac5a0bcb329583f9b',
 'survey_transparency': 4,
 'survey_warmth': 4,
 'survey_theory_of_mind': 4,
 'survey_adaptability': 4,
 'survey_expertise': 4,
 'survey_goals': 7,
 'survey_satisfaction': 7,
 'survey_conflict_resolve': 7,
 'survey_believability': 7,
 'survey_transactivity': 7,
 'survey_truthfulness': 7,
 'survey_additional_feedback': 'thank you',
 'personalit

In [56]:
output_file = 'user_study_hiring_episodes.jsonl'

print(f"Converting {len(data['episodes'])} episodes to JSONL...")

with open(output_file, 'w') as f:
    for i, episode in enumerate(data['episodes']):
        if episode['study_data']['prolific_data'].get('PROLIFIC_PID') is None:
            continue  # Skip episodes without PROLIFIC_PID
        flattened = flatten_episode_complete(episode)
        f.write(json.dumps(flattened) + '\n')
        
        if (i + 1) % 20 == 0:  # Progress indicator
            print(f"Processed {i + 1} episodes...")

print(f"Conversion complete! Saved to {output_file}")

Converting 121 episodes to JSONL...
Processed 20 episodes...
Processed 40 episodes...
Processed 60 episodes...
Processed 80 episodes...
Processed 100 episodes...
Processed 120 episodes...
Conversion complete! Saved to user_study_hiring_episodes.jsonl


In [3]:
conv_df = pd.read_json('user_study_hiring_episodes.jsonl', lines=True)

In [4]:
conv_df.shape

(116, 28)